# 06 - MCP (Model Context Protocol) Integration

이 노트북에서는 **Azure AI Foundry Agent Service**와 **MCP (Model Context Protocol)**를 활용하여
Knowledge Base를 Tool로 연동하는 방법을 학습합니다.

## 📋 학습 내용
1. **MCP 개요** - Model Context Protocol 이해
2. **Azure AI Foundry Agent** - 에이전트 생성 및 설정
3. **Knowledge Base Tool** - KB를 도구로 연결
4. **Agentic RAG 파이프라인** - 에이전트 기반 검색 시스템

## ⚠️ 사전 요구사항
- `01-setup_knowledge_base.ipynb` 실행 완료
- Azure AI Foundry Project 설정

## 1. MCP (Model Context Protocol) 개요

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                   Model Context Protocol (MCP) 개요                           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  🔧 MCP란?                                                                    ║
║  • Anthropic이 제안한 오픈 표준 프로토콜                                       ║
║  • AI 모델과 외부 도구/데이터 소스 연결을 표준화                               ║
║  • JSON-RPC 기반 통신                                                         ║
║                                                                              ║
║  📊 MCP의 핵심 구성요소                                                        ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │                                                                        │  ║
║  │   ┌─────────────┐      ┌─────────────┐      ┌─────────────────────┐   │  ║
║  │   │   Client    │ ──── │ MCP Server  │ ──── │ External Resources  │   │  ║
║  │   │ (AI Agent)  │      │  (Bridge)   │      │ (KB, DB, APIs...)   │   │  ║
║  │   └─────────────┘      └─────────────┘      └─────────────────────┘   │  ║
║  │                                                                        │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
║  🎯 Azure AI Search + MCP 활용                                                ║
║  • Knowledge Base를 MCP Tool로 노출                                           ║
║  • Foundry Agent가 자동으로 KB 검색 수행                                       ║
║  • 복잡한 멀티턴 대화에서 컨텍스트 기반 검색                                   ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

## 2. 환경 설정

In [ ]:
import os
import json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.core.credentials import AzureKeyCredential

# 환경 변수 로드
load_dotenv()

# Azure AI Search 설정
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_api_key = os.environ["SEARCH_ADMIN_KEY"]
search_credential = AzureKeyCredential(search_api_key)

# Azure AI Foundry 설정
foundry_project_endpoint = os.environ.get("AZURE_AI_FOUNDRY_PROJECT_ENDPOINT", "")
foundry_subscription_id = os.environ.get("AZURE_SUBSCRIPTION_ID", "")
foundry_resource_group = os.environ.get("AZURE_RESOURCE_GROUP", "")
foundry_project_name = os.environ.get("AZURE_AI_FOUNDRY_PROJECT_NAME", "")

# Knowledge Base 이름
KNOWLEDGE_BASE_NAME = "products-kb"

print(f"✅ Search Endpoint: {search_endpoint}")
print(f"✅ Foundry Project: {foundry_project_name or 'Not configured'}")
print(f"✅ Knowledge Base: {KNOWLEDGE_BASE_NAME}")

## 3. Azure AI Foundry Agent 클라이언트 설정

In [ ]:
# Azure AI Agent Service SDK 설치 확인
try:
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import (
        Agent,
        AzureAISearchTool,
        AzureAISearchQueryType,
        MessageRole,
        ThreadMessage,
        AgentThread,
    )
    print("✅ Azure AI Projects SDK 로드 완료")
except ImportError:
    print("❌ azure-ai-projects 패키지 설치 필요")
    print("   pip install azure-ai-projects")

In [ ]:
# Foundry Project 클라이언트 생성
credential = DefaultAzureCredential()

if foundry_project_endpoint:
    project_client = AIProjectClient(
        credential=credential,
        endpoint=foundry_project_endpoint
    )
    print(f"✅ AI Project Client 생성 완료")
else:
    print("⚠️ AZURE_AI_FOUNDRY_PROJECT_ENDPOINT 환경 변수를 설정하세요")
    project_client = None

## 4. Knowledge Base Tool 정의

Azure AI Search Knowledge Base를 Agent Tool로 정의합니다.

In [ ]:
# Knowledge Base Tool 설정
def create_knowledge_base_tool():
    """
    Knowledge Base를 Azure AI Search Tool로 정의합니다.
    """
    kb_tool = AzureAISearchTool(
        index_connection_id=f"/subscriptions/{foundry_subscription_id}/resourceGroups/{foundry_resource_group}/providers/Microsoft.Search/searchServices/{search_endpoint.split('//')[1].split('.')[0]}",
        index_name=KNOWLEDGE_BASE_NAME,
        query_type=AzureAISearchQueryType.VECTOR_SEMANTIC_HYBRID,
        top_k=5,
        filter="",
        strictness=3
    )
    return kb_tool

print("✅ Knowledge Base Tool 정의 완료")

## 5. Custom MCP Tool 정의 (Function Calling 방식)

MCP 스타일의 커스텀 도구를 Function Calling 형태로 정의합니다.

In [ ]:
# MCP 스타일 Tool 정의 (JSON Schema)
KNOWLEDGE_RETRIEVAL_TOOL = {
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": "제품 정보 Knowledge Base에서 관련 문서를 검색합니다. 제품 추천, 상세 정보, 가격 비교 등의 질문에 사용합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "검색할 자연어 쿼리"
                },
                "filters": {
                    "type": "object",
                    "description": "검색 필터 (선택사항)",
                    "properties": {
                        "brand": {"type": "string"},
                        "category": {"type": "string"},
                        "price_max": {"type": "number"}
                    }
                },
                "top_k": {
                    "type": "integer",
                    "description": "반환할 최대 결과 수",
                    "default": 5
                }
            },
            "required": ["query"]
        }
    }
}

print("📋 MCP Tool Definition:")
print(json.dumps(KNOWLEDGE_RETRIEVAL_TOOL, indent=2, ensure_ascii=False))

## 6. Tool 실행 함수 (Handler)

In [ ]:
from azure.search.documents import KnowledgeBaseRetrievalClient
from azure.search.documents.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

def execute_knowledge_search(query: str, filters: dict = None, top_k: int = 5) -> dict:
    """
    Knowledge Base 검색을 실행합니다.
    MCP Tool Handler 역할을 수행합니다.
    """
    # KB 클라이언트 생성
    kb_client = KnowledgeBaseRetrievalClient(
        endpoint=search_endpoint,
        knowledge_base_name=KNOWLEDGE_BASE_NAME,
        credential=search_credential
    )
    
    # 필터 문자열 생성
    filter_str = ""
    if filters:
        filter_parts = []
        if filters.get("brand"):
            filter_parts.append(f"brand eq '{filters['brand']}'")
        if filters.get("category"):
            filter_parts.append(f"category eq '{filters['category']}'")
        if filters.get("price_max"):
            filter_parts.append(f"price le {filters['price_max']}")
        filter_str = " and ".join(filter_parts)
    
    # 검색 요청
    request = KnowledgeBaseRetrievalRequest(
        query=query,
        top=top_k,
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA
    )
    
    # 검색 실행
    response = kb_client.retrieve(request)
    
    # 결과 포맷팅
    results = []
    if hasattr(response, 'data') and response.data:
        for doc in response.data:
            results.append({
                "title": doc.get("title", "N/A"),
                "content": doc.get("content", "")[:500],
                "brand": doc.get("brand", "N/A"),
                "price": doc.get("price", "N/A"),
                "score": doc.get("@search.reranker_score", doc.get("@search.score", 0))
            })
    
    return {
        "status": "success",
        "query": query,
        "result_count": len(results),
        "results": results
    }

print("✅ Tool Handler 함수 정의 완료")

In [ ]:
# Tool Handler 테스트
test_result = execute_knowledge_search(
    query="아기 겨울 옷",
    top_k=3
)

print("🔍 Tool 실행 결과:")
print(json.dumps(test_result, indent=2, ensure_ascii=False))

## 7. Agent 생성 및 Tool 연결

In [ ]:
# Agent 시스템 프롬프트 정의
AGENT_SYSTEM_PROMPT = """
당신은 아기 용품 쇼핑 어시스턴트입니다.

## 역할
- 고객의 제품 관련 질문에 답변합니다.
- Knowledge Base에서 관련 제품 정보를 검색합니다.
- 검색 결과를 바탕으로 친절하게 추천합니다.

## 도구 사용 지침
1. 제품 관련 질문이 들어오면 search_knowledge_base 도구를 사용하세요.
2. 검색 결과가 없으면 다른 키워드로 재검색을 시도하세요.
3. 검색 결과를 사용자가 이해하기 쉽게 요약하여 전달하세요.

## 응답 형식
- 한국어로 응답합니다.
- 제품 추천 시 가격, 브랜드, 주요 특징을 포함합니다.
- 여러 옵션이 있으면 비교하여 설명합니다.
"""

print("📋 Agent System Prompt 정의 완료")

In [ ]:
# Agent 생성 (Foundry Agent Service 사용 시)
def create_shopping_agent():
    """
    쇼핑 어시스턴트 Agent를 생성합니다.
    """
    if not project_client:
        print("⚠️ Project Client가 설정되지 않았습니다.")
        return None
    
    agent = project_client.agents.create_agent(
        model="gpt-4o",
        name="shopping-assistant",
        instructions=AGENT_SYSTEM_PROMPT,
        tools=[KNOWLEDGE_RETRIEVAL_TOOL]
    )
    
    print(f"✅ Agent 생성 완료: {agent.id}")
    return agent

# Agent 생성 (프로젝트 클라이언트가 설정된 경우에만)
if project_client:
    shopping_agent = create_shopping_agent()
else:
    print("⚠️ Foundry Project가 설정되지 않아 시뮬레이션 모드로 진행합니다.")
    shopping_agent = None

## 8. Agentic RAG 파이프라인 시뮬레이션

Foundry Agent가 없어도 MCP 스타일의 Agentic RAG 파이프라인을 시뮬레이션할 수 있습니다.

In [ ]:
from openai import AzureOpenAI

# Azure OpenAI 클라이언트 설정
aoai_client = AzureOpenAI(
    api_key=os.environ.get("AOAI_API_KEY", ""),
    api_version="2024-08-01-preview",
    azure_endpoint=os.environ.get("AOAI_ENDPOINT", "")
)

DEPLOYMENT_NAME = os.environ.get("AOAI_DEPLOYMENT_NAME", "gpt-4o")

print(f"✅ Azure OpenAI 클라이언트 설정 완료")
print(f"   Deployment: {DEPLOYMENT_NAME}")

In [ ]:
def run_agentic_rag(user_query: str) -> str:
    """
    MCP 스타일의 Agentic RAG 파이프라인을 실행합니다.
    
    1. LLM이 도구 호출 여부 결정
    2. 도구 실행 (Knowledge Base 검색)
    3. 검색 결과로 최종 응답 생성
    """
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]
    
    tools = [KNOWLEDGE_RETRIEVAL_TOOL]
    
    print(f"\n{'=' * 60}")
    print(f"👤 사용자: {user_query}")
    print(f"{'=' * 60}")
    
    # Step 1: LLM 호출 (도구 선택)
    print("\n🤖 Step 1: LLM이 도구 호출 여부 결정...")
    response = aoai_client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    assistant_message = response.choices[0].message
    
    # Step 2: 도구 호출 처리
    if assistant_message.tool_calls:
        print(f"\n🔧 Step 2: 도구 호출 감지 - {len(assistant_message.tool_calls)}개")
        
        messages.append(assistant_message)
        
        for tool_call in assistant_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"   📞 호출: {function_name}")
            print(f"   📥 파라미터: {json.dumps(function_args, ensure_ascii=False)}")
            
            # MCP Tool 실행
            if function_name == "search_knowledge_base":
                tool_result = execute_knowledge_search(
                    query=function_args.get("query", user_query),
                    filters=function_args.get("filters"),
                    top_k=function_args.get("top_k", 5)
                )
            else:
                tool_result = {"error": f"Unknown tool: {function_name}"}
            
            print(f"   📤 결과: {tool_result['result_count']}개 문서 검색됨")
            
            # 도구 결과를 메시지에 추가
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(tool_result, ensure_ascii=False)
            })
        
        # Step 3: 최종 응답 생성
        print("\n🤖 Step 3: 검색 결과로 최종 응답 생성...")
        final_response = aoai_client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages
        )
        final_answer = final_response.choices[0].message.content
    else:
        print("\n📝 도구 호출 없이 직접 응답")
        final_answer = assistant_message.content
    
    print(f"\n{'=' * 60}")
    print(f"🤖 어시스턴트:")
    print(f"{'=' * 60}")
    print(final_answer)
    
    return final_answer

print("✅ Agentic RAG 파이프라인 정의 완료")

## 9. Agentic RAG 테스트

In [ ]:
# 테스트 1: 제품 추천 질의
result1 = run_agentic_rag("겨울에 아기가 따뜻하게 입을 수 있는 옷을 추천해주세요.")

In [ ]:
# 테스트 2: 가격 조건 질의
result2 = run_agentic_rag("3만원 이하의 아기 용품 중 인기 있는 것은 무엇인가요?")

In [ ]:
# 테스트 3: 브랜드 비교 질의
result3 = run_agentic_rag("블루독베이비와 압소바 브랜드 제품을 비교해주세요.")

In [ ]:
# 테스트 4: 일반 대화 (도구 호출 불필요)
result4 = run_agentic_rag("안녕하세요! 쇼핑몰 영업시간이 어떻게 되나요?")

## 10. 멀티턴 대화 시뮬레이션

In [ ]:
def run_multi_turn_conversation(conversation: list) -> None:
    """
    멀티턴 대화를 시뮬레이션합니다.
    """
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}]
    tools = [KNOWLEDGE_RETRIEVAL_TOOL]
    
    print("\n" + "#" * 70)
    print("🎭 멀티턴 대화 시뮬레이션")
    print("#" * 70)
    
    for turn_num, user_input in enumerate(conversation, 1):
        print(f"\n{'─' * 60}")
        print(f"📍 Turn {turn_num}")
        print(f"{'─' * 60}")
        print(f"👤 사용자: {user_input}")
        
        messages.append({"role": "user", "content": user_input})
        
        # LLM 호출
        response = aoai_client.chat.completions.create(
            model=DEPLOYMENT_NAME,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        assistant_message = response.choices[0].message
        
        # 도구 호출 처리
        if assistant_message.tool_calls:
            print(f"   🔧 도구 호출: search_knowledge_base")
            messages.append(assistant_message)
            
            for tool_call in assistant_message.tool_calls:
                function_args = json.loads(tool_call.function.arguments)
                tool_result = execute_knowledge_search(
                    query=function_args.get("query", user_input),
                    filters=function_args.get("filters"),
                    top_k=function_args.get("top_k", 5)
                )
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result, ensure_ascii=False)
                })
            
            # 최종 응답
            final_response = aoai_client.chat.completions.create(
                model=DEPLOYMENT_NAME,
                messages=messages
            )
            answer = final_response.choices[0].message.content
        else:
            answer = assistant_message.content
        
        messages.append({"role": "assistant", "content": answer})
        print(f"\n🤖 어시스턴트: {answer}")
    
    print(f"\n{'─' * 60}")
    print("✅ 대화 종료")

print("✅ 멀티턴 대화 함수 정의 완료")

In [ ]:
# 멀티턴 대화 테스트
test_conversation = [
    "아기 옷을 찾고 있어요.",
    "그 중에서 겨울용으로 따뜻한 걸 추천해주세요.",
    "가격이 3만원 이하인 것만 보여주세요.",
    "첫 번째 제품에 대해 더 자세히 알려주세요."
]

run_multi_turn_conversation(test_conversation)

## 11. MCP Server 스타일 구현 (참고)

In [ ]:
# MCP Server 스타일의 Tool Registry 구현
class MCPToolRegistry:
    """
    MCP 스타일의 도구 레지스트리입니다.
    실제 MCP Server를 구현할 때 참고할 수 있습니다.
    """
    
    def __init__(self):
        self.tools = {}
        self.handlers = {}
    
    def register_tool(self, name: str, definition: dict, handler: callable):
        """도구를 등록합니다."""
        self.tools[name] = definition
        self.handlers[name] = handler
        print(f"✅ Tool 등록: {name}")
    
    def list_tools(self) -> list:
        """등록된 도구 목록을 반환합니다."""
        return list(self.tools.values())
    
    def call_tool(self, name: str, arguments: dict) -> dict:
        """도구를 실행합니다."""
        if name not in self.handlers:
            return {"error": f"Tool not found: {name}"}
        
        try:
            result = self.handlers[name](**arguments)
            return {"result": result}
        except Exception as e:
            return {"error": str(e)}

# Registry 인스턴스 생성
mcp_registry = MCPToolRegistry()

# Knowledge Base 도구 등록
mcp_registry.register_tool(
    name="search_knowledge_base",
    definition=KNOWLEDGE_RETRIEVAL_TOOL,
    handler=execute_knowledge_search
)

In [ ]:
# MCP Registry를 통한 도구 호출
result = mcp_registry.call_tool(
    name="search_knowledge_base",
    arguments={
        "query": "유아 장난감",
        "top_k": 3
    }
)

print("📋 MCP Registry 도구 호출 결과:")
print(json.dumps(result, indent=2, ensure_ascii=False, default=str)[:1000])

## 12. 정리

이 노트북에서 학습한 내용:

1. **MCP 개요**: Model Context Protocol의 개념과 구조
2. **Tool 정의**: Function Calling 형태의 MCP 도구 정의
3. **Tool Handler**: Knowledge Base 검색 실행 함수
4. **Agentic RAG**: LLM이 자동으로 도구를 선택하고 실행하는 파이프라인
5. **멀티턴 대화**: 컨텍스트를 유지하며 연속 검색
6. **MCP Registry**: 도구 관리 패턴

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                           MCP Integration 요약                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  📋 구현 패턴                                                                  ║
║                                                                              ║
║  1. Tool Definition (도구 정의)                                               ║
║     • JSON Schema로 도구의 입출력 정의                                         ║
║     • 명확한 설명으로 LLM이 적절한 도구 선택 유도                              ║
║                                                                              ║
║  2. Tool Handler (도구 핸들러)                                                 ║
║     • 실제 Knowledge Base 검색 로직 구현                                       ║
║     • 결과를 LLM이 이해할 수 있는 형태로 반환                                  ║
║                                                                              ║
║  3. Agentic RAG Pipeline                                                      ║
║     • LLM → Tool Selection → Tool Execution → Response Generation             ║
║     • 자동화된 검색 및 응답 생성                                               ║
║                                                                              ║
║  🎯 활용 시나리오                                                              ║
║     • 고객 서비스 챗봇                                                         ║
║     • 제품 추천 시스템                                                         ║
║     • 문서 검색 어시스턴트                                                     ║
║     • 복잡한 분석 질의 처리                                                    ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

print("\n✅ MCP Integration 실습 완료!")
print("""
다음 노트북:
- 07-activity_tracing.ipynb: 쿼리 분석 및 디버깅
- 08-answer_synthesis.ipynb: Answer Instructions 커스터마이징
""")